# SigFlow-Sim v4 — Leakage-Controlled, Calibrated, and Auditable

This version keeps the same modelling complexity while tightening the experimental design and preventing blocked network calls from making startup appear frozen.

## What is different

- **Four chronological sections:** training, model validation, calibration, and untouched test.
- **Purging at every boundary:** a sample is not admitted to an earlier section if its future target extends into the next section.
- **No backward filling:** market context is forward-filled for at most three sessions; samples without sufficient past context are discarded.
- **Training-only preprocessing:** winsorisation, scaling, regime thresholds, class weights, transition matrices, and HAR coefficients are fitted from training data only.
- **Per-ticker regimes:** low, medium, and high are defined relative to each asset’s training history.
- **One consistent forward pass:** the shared encoder, gate, quantiles, and all expert parameters are calculated once per batch, so dropout cannot produce contradictory outputs inside one loss.
- **Bounded flow parameters and forecast support:** this prevents numerical explosions while preserving a flexible conditional distribution.
- **Calibration-only tuning:** interval scale and regime-temperature calibration use the calibration section, never validation or test.
- **Fair baselines:** the HAR-style model is fitted separately for each ticker.
- **Cleaner reporting:** pooled, per-ticker, non-overlapping-phase, probabilistic, regime, and paired block-bootstrap results.
- **Explicit provenance:** data providers, date ranges, split cutoffs, skipped samples, configuration, thresholds, calibration parameters, and model weights are saved.

## Fast-start behaviour

Before downloading any symbol, the notebook performs one short connectivity test. If the runtime blocks external internet:

- pipeline-test mode switches to the labelled synthetic fallback within seconds;
- real-experiment mode stops immediately with a clear error;
- it does not wait through every ticker and provider.

Every network attempt is printed with `flush=True`, so the active step is always visible.

## First run

Keep `QUICK_MODE = True`. Use **Restart and Run All**.

For a final real-data experiment, use:

```python
QUICK_MODE = False
experiment_mode = "real_experiment"
allow_synthetic_fallback = False
```

Synthetic data is useful only for confirming that the pipeline runs.

In [ ]:
# Environment diagnostic — no package installation is attempted.

import importlib.util
import sys

REQUIRED_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "torch": "torch",
}

missing = [
    package
    for package, import_name in REQUIRED_IMPORTS.items()
    if importlib.util.find_spec(import_name) is None
]

print("Python executable:", sys.executable)
if missing:
    raise ModuleNotFoundError(
        "Missing core scientific packages: " + ", ".join(missing)
    )

print("Core packages are available.")
print("No yfinance, iisignature, or nflows installation is required.")

In [ ]:
from __future__ import annotations

import copy
import io
import json
import math
import random
import urllib.parse
import urllib.request
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPS = 1e-8
REGIME_NAMES = np.array(["Low", "Medium", "High"])

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Data route: cache → direct Yahoo → Stooq → optional synthetic fallback")

## Configuration

In [ ]:
QUICK_MODE = True

@dataclass(frozen=True)
class Config:
    quick_mode: bool = QUICK_MODE
    experiment_mode: str = (
        "pipeline_test" if QUICK_MODE else "real_experiment"
    )

    # Data and provenance
    use_real_market_data: bool = True
    allow_synthetic_fallback: bool = QUICK_MODE
    refresh_data: bool = False

    # Fast network failure instead of multi-minute apparent hangs
    network_preflight_timeout_seconds: int = 3
    network_request_timeout_seconds: int = 6
    run_network_preflight: bool = True
    tickers: tuple[str, ...] = (
        ("AAPL", "MSFT") if QUICK_MODE
        else ("AAPL", "MSFT", "GOOGL", "AMZN")
    )
    market_symbols: tuple[str, ...] = ("SPY", "QQQ", "^VIX")
    include_market_context: bool = True
    start_date: str = "2020-01-01" if QUICK_MODE else "2010-01-01"
    end_date: str = "2026-07-25"
    cache_dir: str = "sigflow_v4_cache"
    output_dir: str = "sigflow_v4_outputs"

    # Forecast construction
    window: int = 60
    horizon: int = 10
    annualisation: float = 252.0
    ewma_lambda: float = 0.94
    signature_depth: int = 3
    market_forward_fill_limit: int = 3

    # Strict chronological sections
    train_fraction: float = 0.65
    validation_fraction: float = 0.15
    calibration_fraction: float = 0.10

    # Training-only preprocessing
    winsor_lower_quantile: float = 0.001
    winsor_upper_quantile: float = 0.999
    standardised_clip: float = 8.0

    # Model
    regimes: int = 3
    hidden_size: int = 64
    dropout: float = 0.20
    minimum_scale: float = 0.03
    maximum_scale: float = 1.50
    maximum_skew: float = 1.50
    minimum_tail: float = 0.70
    maximum_tail: float = 2.00

    # Training objective
    expert_alignment_weight: float = 0.30
    regime_classification_weight: float = 0.30
    medium_class_multiplier: float = 1.50
    gate_balance_weight: float = 0.005
    qlike_weight: float = 0.05
    quantile_weight: float = 0.10
    label_smoothing: float = 0.02

    # Optimisation
    epochs: int = 20 if QUICK_MODE else 150
    batch_size: int = 128
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    gradient_clip: float = 1.0
    patience: int = 6 if QUICK_MODE else 15
    print_every: int = 1 if QUICK_MODE else 5

    # Ensemble and predictions
    ensemble_seeds: tuple[int, ...] = (
        (42,) if QUICK_MODE else (11, 22, 33)
    )
    prediction_samples: int = 192 if QUICK_MODE else 768
    prediction_batch_size: int = 256
    minimum_reported_volatility: float = 0.01
    maximum_reported_volatility: float = 5.00

    # Calibration-only searches
    interval_scale_min: float = 0.50
    interval_scale_max: float = 2.00
    interval_scale_candidates: int = 76
    temperature_min: float = 0.25
    temperature_max: float = 4.00
    temperature_candidates: int = 151

    # Dependence-aware evaluation
    run_block_bootstrap: bool = True
    bootstrap_repetitions: int = 200 if QUICK_MODE else 1000
    bootstrap_block_days: int = 20

    seed: int = 42


CFG = Config()

if (
    CFG.train_fraction
    + CFG.validation_fraction
    + CFG.calibration_fraction
    >= 1.0
):
    raise ValueError("Train, validation, and calibration fractions must sum below 1.")

if (
    CFG.experiment_mode == "real_experiment"
    and CFG.allow_synthetic_fallback
):
    raise ValueError(
        "Synthetic fallback must be disabled for a final real experiment."
    )

print(json.dumps(asdict(CFG), indent=2))

## Reproducibility, robust preprocessing, and data containers

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        pass

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = False


@dataclass
class RobustStandardiser:
    lower: np.ndarray
    upper: np.ndarray
    mean: np.ndarray
    scale: np.ndarray
    z_clip: float

    @classmethod
    def fit(
        cls,
        values: np.ndarray,
        lower_quantile: float,
        upper_quantile: float,
        z_clip: float,
    ) -> "RobustStandardiser":
        lower = np.nanquantile(values, lower_quantile, axis=0)
        upper = np.nanquantile(values, upper_quantile, axis=0)
        upper = np.maximum(upper, lower)

        clipped = np.clip(values, lower, upper)
        mean = np.nanmean(clipped, axis=0)
        scale = np.nanstd(clipped, axis=0)
        scale = np.where(
            np.isfinite(scale) & (scale > 1e-6),
            scale,
            1.0,
        )

        return cls(
            lower=lower.astype(np.float32),
            upper=upper.astype(np.float32),
            mean=mean.astype(np.float32),
            scale=scale.astype(np.float32),
            z_clip=float(z_clip),
        )

    def transform(self, values: np.ndarray) -> np.ndarray:
        clipped = np.clip(values, self.lower, self.upper)
        transformed = (clipped - self.mean) / self.scale
        transformed = np.clip(
            transformed,
            -self.z_clip,
            self.z_clip,
        )
        return np.nan_to_num(
            transformed,
            nan=0.0,
            posinf=self.z_clip,
            neginf=-self.z_clip,
        ).astype(np.float32)


@dataclass
class MarketDataset:
    features: np.ndarray
    targets_log_vol: np.ndarray
    metadata: pd.DataFrame
    feature_names: list[str]
    data_source: str
    available_market_symbols: tuple[str, ...]
    data_manifest: pd.DataFrame
    skipped_samples: pd.DataFrame


@dataclass
class PreparedSplit:
    train_indices: np.ndarray
    validation_indices: np.ndarray
    calibration_indices: np.ndarray
    test_indices: np.ndarray
    context: np.ndarray
    regime_labels: np.ndarray
    ticker_thresholds: np.ndarray
    class_weights: np.ndarray
    train_regime_proportions: np.ndarray
    transition_matrices: np.ndarray
    initial_regime_probabilities: np.ndarray
    standardiser: RobustStandardiser
    train_cutoff: pd.Timestamp
    validation_cutoff: pd.Timestamp
    calibration_cutoff: pd.Timestamp
    leakage_audit: pd.DataFrame


set_seed(CFG.seed)

## Dependency-free truncated path signature

The three path channels are normalised time, cumulative standardised return, and cumulative quadratic variation. A depth-three signature contributes \(3+3^2+3^3=39\) features.

In [ ]:
def truncated_signature(path: np.ndarray, depth: int = 3) -> np.ndarray:
    if depth not in (1, 2, 3):
        raise ValueError("Supported signature depths are 1, 2, and 3.")

    increments = np.diff(np.asarray(path, dtype=np.float64), axis=0)
    dimension = path.shape[1]

    level1 = np.zeros(dimension, dtype=np.float64)
    level2 = np.zeros((dimension, dimension), dtype=np.float64)
    level3 = np.zeros(
        (dimension, dimension, dimension),
        dtype=np.float64,
    )

    for increment in increments:
        segment1 = increment
        segment2 = np.einsum(
            "i,j->ij", increment, increment
        ) / 2.0
        segment3 = np.einsum(
            "i,j,k->ijk",
            increment,
            increment,
            increment,
        ) / 6.0

        old1 = level1.copy()
        old2 = level2.copy()

        level1 = old1 + segment1
        if depth >= 2:
            level2 = (
                old2
                + np.einsum("i,j->ij", old1, segment1)
                + segment2
            )
        if depth >= 3:
            level3 = (
                level3
                + np.einsum("ij,k->ijk", old2, segment1)
                + np.einsum("i,jk->ijk", old1, segment2)
                + segment3
            )

    levels = [level1.ravel()]
    if depth >= 2:
        levels.append(level2.ravel())
    if depth >= 3:
        levels.append(level3.ravel())
    return np.concatenate(levels)


def signature_feature_names(
    path_dimension: int,
    depth: int,
) -> list[str]:
    names = []
    for level in range(1, depth + 1):
        for index in range(path_dimension**level):
            names.append(f"signature_L{level}_{index}")
    return names

## Direct OHLCV loading with provenance and no invented prices

In [ ]:
REQUIRED_PRICE_COLUMNS = ["Open", "High", "Low", "Close"]


def clean_ohlcv(frame: pd.DataFrame, symbol: str) -> pd.DataFrame:
    frame = frame.copy()
    frame.index = pd.to_datetime(frame.index, errors="coerce")
    if getattr(frame.index, "tz", None) is not None:
        frame.index = frame.index.tz_localize(None)

    frame = frame[~frame.index.isna()]
    for column in REQUIRED_PRICE_COLUMNS + ["Volume"]:
        if column not in frame:
            frame[column] = np.nan
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    # Never replace missing Open/High/Low with Close. Those rows are dropped.
    frame = frame.dropna(subset=REQUIRED_PRICE_COLUMNS)
    frame = frame[
        (frame["Open"] > 0)
        & (frame["High"] > 0)
        & (frame["Low"] > 0)
        & (frame["Close"] > 0)
    ]

    frame["VolumeMissing"] = frame["Volume"].isna().astype(float)
    frame["Volume"] = frame["Volume"].fillna(0.0)

    frame = frame[
        ["Open", "High", "Low", "Close", "Volume", "VolumeMissing"]
    ]
    frame = frame.replace([np.inf, -np.inf], np.nan)
    frame = frame.dropna(subset=REQUIRED_PRICE_COLUMNS)
    frame = frame[~frame.index.duplicated(keep="last")].sort_index()

    if frame.empty:
        raise ValueError(f"No usable OHLCV observations for {symbol}.")
    return frame


def request_bytes(
    url: str,
    timeout: int,
) -> bytes:
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 Chrome/124 Safari/537.36"
            ),
            "Accept": "application/json,text/csv,*/*",
        },
    )
    with urllib.request.urlopen(
        request,
        timeout=timeout,
    ) as response:
        return response.read()


def network_preflight(cfg: Config) -> tuple[bool, str]:
    if not cfg.run_network_preflight:
        return True, "preflight disabled"

    url = (
        "https://query1.finance.yahoo.com/v8/finance/chart/"
        "AAPL?range=5d&interval=1d"
    )
    print(
        "Checking external market-data connectivity "
        f"(timeout={cfg.network_preflight_timeout_seconds}s)...",
        flush=True,
    )

    try:
        payload = request_bytes(
            url,
            timeout=cfg.network_preflight_timeout_seconds,
        )
        if not payload:
            return False, "empty response"
        print("  Connectivity check succeeded.", flush=True)
        return True, "ok"
    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"
        print(
            "  Connectivity check failed quickly: " + message,
            flush=True,
        )
        return False, message


def download_yahoo_frame(
    symbol: str,
    start_date: str,
    end_date: str,
    timeout: int,
) -> pd.DataFrame:
    period1 = int(pd.Timestamp(start_date, tz="UTC").timestamp())
    period2 = int(pd.Timestamp(end_date, tz="UTC").timestamp())

    parameters = urllib.parse.urlencode({
        "period1": period1,
        "period2": period2,
        "interval": "1d",
        "events": "history",
        "includeAdjustedClose": "true",
    })
    quoted_symbol = urllib.parse.quote(symbol, safe="")
    url = (
        "https://query1.finance.yahoo.com/v8/finance/chart/"
        f"{quoted_symbol}?{parameters}"
    )

    payload = json.loads(
        request_bytes(url, timeout=timeout).decode("utf-8")
    )
    chart = payload.get("chart", {})
    if chart.get("error"):
        raise RuntimeError(str(chart["error"]))

    results = chart.get("result")
    if not results:
        raise ValueError(f"Yahoo returned no result for {symbol}.")

    result = results[0]
    timestamps = result.get("timestamp") or []
    quote_blocks = result.get("indicators", {}).get("quote") or []
    if not timestamps or not quote_blocks:
        raise ValueError(f"Yahoo returned incomplete OHLCV for {symbol}.")

    quote = quote_blocks[0]
    dates = pd.to_datetime(
        timestamps,
        unit="s",
        utc=True,
    ).tz_localize(None)

    frame = pd.DataFrame({
        "Open": quote.get("open"),
        "High": quote.get("high"),
        "Low": quote.get("low"),
        "Close": quote.get("close"),
        "Volume": quote.get("volume"),
    }, index=dates)

    adjusted_blocks = (
        result.get("indicators", {}).get("adjclose") or []
    )
    adjusted = (
        adjusted_blocks[0].get("adjclose")
        if adjusted_blocks else None
    )

    if adjusted is not None:
        raw_close = pd.to_numeric(
            frame["Close"], errors="coerce"
        )
        adjustment = (
            pd.Series(adjusted, index=dates) / raw_close
        )
        adjustment = adjustment.replace(
            [np.inf, -np.inf], np.nan
        )

        for column in REQUIRED_PRICE_COLUMNS:
            frame[column] = (
                pd.to_numeric(frame[column], errors="coerce")
                * adjustment
            )

    return clean_ohlcv(frame, symbol)


def stooq_symbol(symbol: str) -> str:
    if symbol == "^VIX":
        return "vix"
    return symbol.lower().replace(".", "-") + ".us"


def download_stooq_frame(
    symbol: str,
    start_date: str,
    end_date: str,
    timeout: int,
) -> pd.DataFrame:
    parameters = urllib.parse.urlencode({
        "s": stooq_symbol(symbol),
        "d1": pd.Timestamp(start_date).strftime("%Y%m%d"),
        "d2": pd.Timestamp(end_date).strftime("%Y%m%d"),
        "i": "d",
    })
    raw = request_bytes(
        f"https://stooq.com/q/d/l/?{parameters}",
        timeout=timeout,
    )
    frame = pd.read_csv(io.BytesIO(raw))

    if frame.empty or "Date" not in frame or "Close" not in frame:
        preview = raw[:150].decode("utf-8", errors="replace")
        raise ValueError(f"Stooq returned no useful data: {preview!r}")

    frame.index = pd.to_datetime(frame.pop("Date"), errors="coerce")
    return clean_ohlcv(frame, symbol)


def synthetic_frames(cfg: Config) -> dict[str, pd.DataFrame]:
    dates = pd.date_range(
        cfg.start_date,
        cfg.end_date,
        freq="B",
        inclusive="left",
    )
    rng = np.random.default_rng(cfg.seed + 9000)

    regimes = np.zeros(len(dates), dtype=int)
    transition = np.array([
        [0.975, 0.023, 0.002],
        [0.030, 0.940, 0.030],
        [0.010, 0.080, 0.910],
    ])
    for index in range(1, len(dates)):
        regimes[index] = rng.choice(
            3,
            p=transition[regimes[index - 1]],
        )

    common_volatility = np.array(
        [0.007, 0.014, 0.030]
    )[regimes]
    market_return = rng.normal(0.0002, common_volatility)

    frames = {}
    symbols = list(cfg.tickers) + list(cfg.market_symbols)

    for symbol_id, symbol in enumerate(symbols):
        local_rng = np.random.default_rng(
            cfg.seed + 1000 * (symbol_id + 1)
        )

        if symbol == "^VIX":
            close = (
                13.0
                + 7.0 * regimes
                + 250.0 * np.abs(market_return)
                + local_rng.normal(0.0, 1.4, len(dates))
            )
            close = np.maximum(close, 8.0)
            open_price = close * np.exp(
                local_rng.normal(0.0, 0.01, len(dates))
            )
        else:
            beta = 0.75 + 0.12 * symbol_id
            idiosyncratic = local_rng.normal(
                0.0001,
                np.array([0.006, 0.010, 0.020])[regimes],
            )
            returns = beta * market_return + idiosyncratic
            close = 100.0 * np.exp(np.cumsum(returns))
            previous_close = np.concatenate([[close[0]], close[:-1]])
            open_price = previous_close * np.exp(
                local_rng.normal(
                    0.0,
                    common_volatility * 0.25,
                )
            )

        range_scale = (
            0.004
            + 0.8 * np.abs(
                np.log(close / np.maximum(open_price, EPS))
            )
            + 0.5 * common_volatility
        )
        high = np.maximum(open_price, close) * np.exp(
            range_scale / 2.0
        )
        low = np.minimum(open_price, close) * np.exp(
            -range_scale / 2.0
        )
        volume = local_rng.lognormal(
            mean=16.0 + 7.0 * common_volatility,
            sigma=0.35,
            size=len(dates),
        )

        frames[symbol] = clean_ohlcv(
            pd.DataFrame({
                "Open": open_price,
                "High": high,
                "Low": low,
                "Close": close,
                "Volume": volume,
            }, index=dates),
            symbol,
        )

    return frames


def cache_path_for(symbol: str, cfg: Config) -> Path:
    safe_symbol = (
        symbol.replace("^", "INDEX_").replace("/", "_")
    )
    directory = Path(cfg.cache_dir)
    directory.mkdir(parents=True, exist_ok=True)
    return directory / (
        f"{safe_symbol}_{cfg.start_date}_{cfg.end_date}.csv"
    )


def load_cached_frame(
    symbol: str,
    cfg: Config,
) -> pd.DataFrame | None:
    path = cache_path_for(symbol, cfg)
    if not path.exists() or cfg.refresh_data:
        return None

    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    if frame.empty:
        return None

    print(f"  {symbol}: loaded cache")
    return clean_ohlcv(frame, symbol)


def download_symbol(
    symbol: str,
    cfg: Config,
) -> tuple[pd.DataFrame, str]:
    cached = load_cached_frame(symbol, cfg)
    if cached is not None:
        return cached, "cache"

    errors = []
    providers = [
        ("yahoo_direct", download_yahoo_frame),
        ("stooq_direct", download_stooq_frame),
    ]

    for provider, downloader in providers:
        print(
            f"  {symbol}: trying {provider} "
            f"(timeout={cfg.network_request_timeout_seconds}s)...",
            flush=True,
        )
        try:
            frame = downloader(
                symbol,
                cfg.start_date,
                cfg.end_date,
                cfg.network_request_timeout_seconds,
            )
            frame.to_csv(cache_path_for(symbol, cfg))
            print(
                f"  {symbol}: {len(frame):,} rows via {provider}",
                flush=True,
            )
            return frame, provider
        except Exception as exc:
            message = (
                f"{provider}: {type(exc).__name__}: {exc}"
            )
            errors.append(message)
            print(f"  {symbol}: {message}", flush=True)

    raise RuntimeError(" | ".join(errors))


def build_manifest(
    frames: dict[str, pd.DataFrame],
    providers: dict[str, str],
) -> pd.DataFrame:
    rows = []
    for symbol, frame in frames.items():
        rows.append({
            "symbol": symbol,
            "provider": providers.get(symbol, "unknown"),
            "rows": len(frame),
            "first_date": frame.index.min(),
            "last_date": frame.index.max(),
            "missing_volume_rows": int(
                frame["VolumeMissing"].sum()
            ),
        })
    return pd.DataFrame(rows)


def load_all_frames(
    cfg: Config,
) -> tuple[
    dict[str, pd.DataFrame],
    tuple[str, ...],
    str,
    pd.DataFrame,
]:
    if not cfg.use_real_market_data:
        print(
            "Real market-data requests are disabled; "
            "using synthetic pipeline-test data immediately.",
            flush=True,
        )
        frames = synthetic_frames(cfg)
        providers = {
            symbol: "synthetic_requested"
            for symbol in frames
        }
        return (
            frames,
            cfg.market_symbols,
            "synthetic_requested",
            build_manifest(frames, providers),
        )

    # Cached target data can be used without an internet connection.
    target_cache_complete = all(
        cache_path_for(symbol, cfg).exists()
        and not cfg.refresh_data
        for symbol in cfg.tickers
    )

    if not target_cache_complete:
        connected, reason = network_preflight(cfg)
        if not connected:
            if not cfg.allow_synthetic_fallback:
                raise ConnectionError(
                    "The runtime cannot reach the market-data endpoint. "
                    "Real-experiment mode will not substitute synthetic data. "
                    f"Preflight result: {reason}"
                )

            print(
                "WARNING: external internet is unavailable. "
                "Switching immediately to the clearly labelled "
                "synthetic pipeline-test dataset.",
                flush=True,
            )
            frames = synthetic_frames(cfg)
            providers = {
                symbol: "synthetic_fallback_fast"
                for symbol in frames
            }
            return (
                frames,
                cfg.market_symbols,
                "synthetic_fallback_fast",
                build_manifest(frames, providers),
            )
    else:
        print(
            "Target caches are available; skipping connectivity preflight.",
            flush=True,
        )

    frames = {}
    providers = {}

    try:
        print("Loading target assets...", flush=True)
        for symbol in cfg.tickers:
            frame, provider = download_symbol(symbol, cfg)
            frames[symbol] = frame
            providers[symbol] = provider
    except Exception as exc:
        if not cfg.allow_synthetic_fallback:
            raise

        print(
            "\nTarget market-data loading failed: "
            f"{type(exc).__name__}: {exc}",
            flush=True,
        )
        print(
            "WARNING: switching to synthetic pipeline-test data.",
            flush=True,
        )
        frames = synthetic_frames(cfg)
        providers = {
            symbol: "synthetic_fallback"
            for symbol in frames
        }
        return (
            frames,
            cfg.market_symbols,
            "synthetic_fallback",
            build_manifest(frames, providers),
        )

    available_market_symbols = []
    if cfg.include_market_context:
        print("Loading market context...", flush=True)
        for symbol in cfg.market_symbols:
            try:
                frame, provider = download_symbol(symbol, cfg)
                frames[symbol] = frame
                providers[symbol] = provider
                available_market_symbols.append(symbol)
            except Exception as exc:
                print(
                    f"  {symbol}: omitted because loading failed: {exc}",
                    flush=True,
                )

    source_name = "real_" + "+".join(
        sorted(set(providers.values()))
    )
    return (
        frames,
        tuple(available_market_symbols),
        source_name,
        build_manifest(frames, providers),
    )


## Feature engineering, forward-only alignment, and dataset construction

In [ ]:
STATISTIC_NAMES = [
    "annualised_mean_return",
    "log_annualised_volatility",
    "mean_absolute_return",
    "downside_annualised_volatility",
    "upside_annualised_volatility",
    "maximum_absolute_return",
    "last_return",
    "cumulative_return",
    "lag1_return_autocorrelation",
    "skewness",
    "excess_kurtosis",
    "volatility_of_volatility",
    "squared_return_trend",
    "log_realised_volatility_5",
    "log_realised_volatility_10",
    "log_realised_volatility_20",
    "log_realised_volatility_60",
    "volatility_ratio_5_20",
    "volatility_ratio_20_60",
    "absolute_return_autocorrelation",
    "squared_return_autocorrelation",
    "large_return_fraction",
    "maximum_drawdown",
    "current_return_streak",
]

OHLCV_FEATURE_NAMES = [
    "mean_log_intraday_range",
    "last_log_intraday_range",
    "log_parkinson_volatility_5",
    "log_parkinson_volatility_20",
    "log_parkinson_volatility_60",
    "mean_absolute_overnight_gap",
    "last_overnight_gap",
    "last_log_relative_volume",
    "mean_log_relative_volume",
    "volume_volatility",
    "missing_volume_fraction",
]


def safe_autocorrelation(values: np.ndarray) -> float:
    if len(values) < 3:
        return 0.0

    left = values[:-1]
    right = values[1:]
    if np.std(left) < EPS or np.std(right) < EPS:
        return 0.0

    result = np.corrcoef(left, right)[0, 1]
    return float(result) if np.isfinite(result) else 0.0


def recent_volatility(
    values: np.ndarray,
    length: int,
    annualisation: float,
) -> float:
    selected = values[-min(length, len(values)):]
    return float(
        np.sqrt(annualisation * np.mean(selected**2))
    )


def volatility_of_volatility(
    values: np.ndarray,
    subwindow: int = 5,
) -> float:
    rolling_mean_square = np.convolve(
        values**2,
        np.ones(subwindow) / subwindow,
        mode="valid",
    )
    return float(
        np.std(np.sqrt(np.maximum(rolling_mean_square, 0.0)))
    )


def maximum_drawdown_from_returns(
    values: np.ndarray,
) -> float:
    cumulative_prices = np.exp(np.cumsum(values))
    running_maximum = np.maximum.accumulate(cumulative_prices)
    return float(
        np.min(cumulative_prices / running_maximum - 1.0)
    )


def current_streak(values: np.ndarray) -> float:
    if len(values) == 0 or values[-1] == 0:
        return 0.0

    final_sign = np.sign(values[-1])
    length = 0
    for value in values[::-1]:
        if np.sign(value) == final_sign:
            length += 1
        else:
            break

    return float(length * final_sign)


def statistical_features(
    past: np.ndarray,
    annualisation: float,
) -> np.ndarray:
    mean_return = float(np.mean(past))
    sample_std = max(float(np.std(past, ddof=1)), EPS)
    annualised_volatility = (
        math.sqrt(annualisation) * sample_std
    )

    downside = np.minimum(past, 0.0)
    upside = np.maximum(past, 0.0)
    standardised = (past - mean_return) / sample_std

    time = np.arange(len(past), dtype=float)
    centred_time = time - time.mean()
    denominator = max(
        float(np.sum(centred_time**2)),
        EPS,
    )
    squared_return_trend = float(
        np.sum(
            centred_time
            * (past**2 - np.mean(past**2))
        )
        / denominator
    )

    vol5 = recent_volatility(past, 5, annualisation)
    vol10 = recent_volatility(past, 10, annualisation)
    vol20 = recent_volatility(past, 20, annualisation)
    vol60 = recent_volatility(past, 60, annualisation)

    large_return_fraction = float(
        np.mean(
            np.abs(past - mean_return)
            > 2.0 * sample_std
        )
    )

    return np.array([
        annualisation * mean_return,
        math.log(annualised_volatility + EPS),
        np.mean(np.abs(past)),
        math.sqrt(
            annualisation * np.mean(downside**2)
        ),
        math.sqrt(
            annualisation * np.mean(upside**2)
        ),
        np.max(np.abs(past)),
        past[-1],
        np.sum(past),
        safe_autocorrelation(past),
        np.mean(standardised**3),
        np.mean(standardised**4) - 3.0,
        volatility_of_volatility(past),
        squared_return_trend,
        math.log(vol5 + EPS),
        math.log(vol10 + EPS),
        math.log(vol20 + EPS),
        math.log(vol60 + EPS),
        vol5 / (vol20 + EPS),
        vol20 / (vol60 + EPS),
        safe_autocorrelation(np.abs(past)),
        safe_autocorrelation(past**2),
        large_return_fraction,
        maximum_drawdown_from_returns(past),
        current_streak(past),
    ], dtype=np.float64)


def ohlcv_features(
    past_frame: pd.DataFrame,
    annualisation: float,
) -> np.ndarray:
    open_price = past_frame["Open"].to_numpy(dtype=float)
    high = past_frame["High"].to_numpy(dtype=float)
    low = past_frame["Low"].to_numpy(dtype=float)
    close = past_frame["Close"].to_numpy(dtype=float)
    volume = past_frame["Volume"].to_numpy(dtype=float)
    volume_missing = past_frame[
        "VolumeMissing"
    ].to_numpy(dtype=float)

    log_range = np.log(
        np.maximum(high, EPS) / np.maximum(low, EPS)
    )
    previous_close = np.concatenate(
        [[close[0]], close[:-1]]
    )
    overnight_gap = np.log(
        np.maximum(open_price, EPS)
        / np.maximum(previous_close, EPS)
    )

    def parkinson(length: int) -> float:
        selected = log_range[-min(length, len(log_range)):]
        variance = (
            np.mean(selected**2)
            / (4.0 * math.log(2.0))
        )
        return math.sqrt(
            annualisation * max(variance, EPS)
        )

    volume_series = pd.Series(volume)
    rolling_volume = volume_series.rolling(
        20,
        min_periods=5,
    ).mean().to_numpy()

    positive_volume = volume[volume > 0]
    fallback_volume = (
        float(np.median(positive_volume))
        if len(positive_volume) else 1.0
    )
    rolling_volume = np.where(
        np.isfinite(rolling_volume)
        & (rolling_volume > 0),
        rolling_volume,
        fallback_volume,
    )
    log_relative_volume = np.log(
        (volume + 1.0) / (rolling_volume + 1.0)
    )

    return np.array([
        np.mean(log_range),
        log_range[-1],
        math.log(parkinson(5) + EPS),
        math.log(parkinson(20) + EPS),
        math.log(parkinson(60) + EPS),
        np.mean(np.abs(overnight_gap)),
        overnight_gap[-1],
        log_relative_volume[-1],
        np.mean(log_relative_volume),
        np.std(log_relative_volume),
        np.mean(volume_missing),
    ], dtype=np.float64)


def market_feature_names(
    symbols: tuple[str, ...],
) -> list[str]:
    names = []
    for symbol in symbols:
        safe_symbol = symbol.lower().replace("^", "")
        if symbol == "^VIX":
            names.extend([
                f"{safe_symbol}_level",
                f"{safe_symbol}_change_5",
                f"{safe_symbol}_mean_20",
                f"{safe_symbol}_volatility_20",
            ])
        else:
            names.extend([
                f"{safe_symbol}_log_volatility_5",
                f"{safe_symbol}_log_volatility_20",
                f"{safe_symbol}_log_volatility_60",
                f"{safe_symbol}_return_5",
                f"{safe_symbol}_return_20",
                f"{safe_symbol}_downside_volatility_20",
            ])
    return names


def align_market_data_forward_only(
    target_dates: pd.DatetimeIndex,
    frames: dict[str, pd.DataFrame],
    symbols: tuple[str, ...],
    forward_fill_limit: int,
) -> dict[str, dict[str, np.ndarray]]:
    aligned = {}

    for symbol in symbols:
        raw_close = frames[symbol]["Close"].reindex(target_dates)
        close = raw_close.ffill(limit=forward_fill_limit)

        # No backward fill is permitted.
        returns = np.log(close).diff()

        aligned[symbol] = {
            "close": close.to_numpy(dtype=float),
            "returns": returns.to_numpy(dtype=float),
            "originally_missing": raw_close.isna().to_numpy(),
        }

    return aligned


def market_features_or_none(
    aligned_market_data: dict[str, dict[str, np.ndarray]],
    end_index: int,
    symbols: tuple[str, ...],
    cfg: Config,
) -> np.ndarray | None:
    values = []

    for symbol in symbols:
        information = aligned_market_data[symbol]
        close = information["close"][
            end_index - cfg.window:end_index
        ]
        returns = information["returns"][
            end_index - cfg.window:end_index
        ]

        if (
            len(close) != cfg.window
            or len(returns) != cfg.window
            or not np.all(np.isfinite(close))
            or not np.all(np.isfinite(returns))
        ):
            return None

        if symbol == "^VIX":
            values.extend([
                close[-1],
                close[-1] / (close[-6] + EPS) - 1.0,
                np.mean(close[-20:]),
                np.std(close[-20:]),
            ])
        else:
            downside = np.minimum(returns[-20:], 0.0)
            values.extend([
                math.log(
                    recent_volatility(
                        returns,
                        5,
                        cfg.annualisation,
                    )
                    + EPS
                ),
                math.log(
                    recent_volatility(
                        returns,
                        20,
                        cfg.annualisation,
                    )
                    + EPS
                ),
                math.log(
                    recent_volatility(
                        returns,
                        60,
                        cfg.annualisation,
                    )
                    + EPS
                ),
                np.sum(returns[-5:]),
                np.sum(returns[-20:]),
                math.sqrt(
                    cfg.annualisation
                    * np.mean(downside**2)
                ),
            ])

    return np.asarray(values, dtype=np.float64)


def make_signature_path(past: np.ndarray) -> np.ndarray:
    mean = np.mean(past)
    scale = max(float(np.std(past, ddof=1)), EPS)
    standardised = (past - mean) / scale

    time = np.linspace(0.0, 1.0, len(past) + 1)
    cumulative = np.concatenate([
        [0.0],
        np.cumsum(standardised) / math.sqrt(len(past)),
    ])
    quadratic_variation = np.concatenate([
        [0.0],
        np.cumsum(standardised**2) / len(past),
    ])

    return np.column_stack([
        time,
        cumulative,
        quadratic_variation,
    ])


def realised_volatility(
    values: np.ndarray,
    annualisation: float,
) -> float:
    return float(
        np.sqrt(annualisation * np.mean(values**2))
    )


def ewma_volatility(
    values: np.ndarray,
    annualisation: float,
    decay: float,
) -> float:
    variance = max(float(np.var(values, ddof=1)), EPS)

    for value in values:
        variance = (
            decay * variance
            + (1.0 - decay) * float(value**2)
        )

    return math.sqrt(
        annualisation * max(variance, EPS)
    )


def build_market_dataset(cfg: Config) -> MarketDataset:
    (
        frames,
        available_market_symbols,
        data_source,
        data_manifest,
    ) = load_all_frames(cfg)

    if (
        cfg.experiment_mode == "real_experiment"
        and data_source.startswith("synthetic")
    ):
        raise RuntimeError(
            "A final real experiment may not use synthetic data."
        )

    feature_rows = []
    target_rows = []
    metadata_rows = []
    skipped_rows = []

    feature_names = (
        signature_feature_names(3, cfg.signature_depth)
        + STATISTIC_NAMES
        + OHLCV_FEATURE_NAMES
        + market_feature_names(available_market_symbols)
    )

    print("\nBuilding leakage-controlled samples...")

    for ticker_id, ticker in enumerate(cfg.tickers):
        frame = frames[ticker].copy()

        close_returns = np.log(
            frame["Close"]
        ).diff().dropna()
        dates = pd.DatetimeIndex(close_returns.index)
        returns = close_returns.to_numpy(dtype=np.float64)
        aligned_asset_frame = frame.reindex(dates)

        aligned_markets = align_market_data_forward_only(
            dates,
            frames,
            available_market_symbols,
            cfg.market_forward_fill_limit,
        )

        accepted = 0
        skipped = 0

        for index in range(
            cfg.window,
            len(returns) - cfg.horizon + 1,
        ):
            past = returns[index - cfg.window:index]
            future = returns[index:index + cfg.horizon]
            past_frame = aligned_asset_frame.iloc[
                index - cfg.window:index
            ]

            if past_frame[REQUIRED_PRICE_COLUMNS].isna().any().any():
                skipped_rows.append({
                    "ticker": ticker,
                    "origin_date": dates[index - 1],
                    "reason": "missing_target_ohlc",
                })
                skipped += 1
                continue

            market_values = market_features_or_none(
                aligned_markets,
                index,
                available_market_symbols,
                cfg,
            )
            if market_values is None:
                skipped_rows.append({
                    "ticker": ticker,
                    "origin_date": dates[index - 1],
                    "reason": "insufficient_forward_only_market_context",
                })
                skipped += 1
                continue

            feature = np.concatenate([
                truncated_signature(
                    make_signature_path(past),
                    cfg.signature_depth,
                ),
                statistical_features(
                    past,
                    cfg.annualisation,
                ),
                ohlcv_features(
                    past_frame,
                    cfg.annualisation,
                ),
                market_values,
            ])

            actual_volatility = realised_volatility(
                future,
                cfg.annualisation,
            )

            feature_rows.append(feature)
            target_rows.append(
                math.log(actual_volatility + EPS)
            )
            metadata_rows.append({
                "ticker_id": ticker_id,
                "ticker": ticker,
                "origin_date": dates[index - 1],
                "target_start_date": dates[index],
                "target_end_date": dates[
                    index + cfg.horizon - 1
                ],
                "actual_vol": actual_volatility,
                "rolling_vol_baseline": recent_volatility(
                    past,
                    20,
                    cfg.annualisation,
                ),
                "ewma_vol_baseline": ewma_volatility(
                    past,
                    cfg.annualisation,
                    cfg.ewma_lambda,
                ),
                "data_source": data_source,
            })
            accepted += 1

        print(
            f"  {ticker}: accepted={accepted:,}, skipped={skipped:,}"
        )

    if not feature_rows:
        raise RuntimeError("No samples survived the leakage controls.")

    unsorted_metadata = pd.DataFrame(metadata_rows)
    ordering = (
        unsorted_metadata.assign(
            _position=np.arange(len(unsorted_metadata))
        )
        .sort_values(["origin_date", "ticker"])["_position"]
        .to_numpy(dtype=int)
    )

    metadata = (
        unsorted_metadata
        .iloc[ordering]
        .reset_index(drop=True)
    )
    features = np.asarray(
        feature_rows,
        dtype=np.float32,
    )[ordering]
    targets = np.asarray(
        target_rows,
        dtype=np.float32,
    )[ordering]

    if features.shape[1] != len(feature_names):
        raise RuntimeError(
            f"Feature mismatch: values={features.shape[1]}, "
            f"names={len(feature_names)}"
        )

    if not np.all(np.isfinite(features)):
        raise ValueError("Non-finite feature values remain.")

    if not np.all(np.isfinite(targets)):
        raise ValueError("Non-finite target values remain.")

    skipped_samples = pd.DataFrame(
        skipped_rows,
        columns=["ticker", "origin_date", "reason"],
    )

    print(
        f"Dataset complete: {len(metadata):,} samples, "
        f"{features.shape[1]} features, source={data_source}"
    )

    return MarketDataset(
        features=features,
        targets_log_vol=targets,
        metadata=metadata,
        feature_names=feature_names,
        data_source=data_source,
        available_market_symbols=available_market_symbols,
        data_manifest=data_manifest,
        skipped_samples=skipped_samples,
    )

## Four-way purged split and training-only fitting

In [ ]:
def cutoff_from_fraction(
    unique_dates: np.ndarray,
    fraction: float,
) -> pd.Timestamp:
    position = int(len(unique_dates) * fraction) - 1
    position = max(0, min(position, len(unique_dates) - 1))
    return pd.Timestamp(unique_dates[position])


def build_ticker_thresholds_and_labels(
    dataset: MarketDataset,
    train_indices: np.ndarray,
    cfg: Config,
) -> tuple[np.ndarray, np.ndarray]:
    metadata = dataset.metadata
    thresholds = np.zeros(
        (len(cfg.tickers), 2),
        dtype=np.float32,
    )
    labels = np.zeros(len(metadata), dtype=np.int64)

    for ticker_id, ticker in enumerate(cfg.tickers):
        ticker_train_indices = train_indices[
            metadata.iloc[train_indices][
                "ticker_id"
            ].to_numpy(dtype=int)
            == ticker_id
        ]

        if len(ticker_train_indices) < 30:
            raise ValueError(
                f"Too few training samples for {ticker}."
            )

        ticker_thresholds = np.quantile(
            dataset.targets_log_vol[ticker_train_indices],
            [1 / 3, 2 / 3],
        )
        thresholds[ticker_id] = ticker_thresholds

        ticker_indices = np.flatnonzero(
            metadata["ticker_id"].to_numpy(dtype=int)
            == ticker_id
        )
        labels[ticker_indices] = np.digitize(
            dataset.targets_log_vol[ticker_indices],
            ticker_thresholds,
        )

    return thresholds, labels


def build_training_regime_statistics(
    metadata: pd.DataFrame,
    labels: np.ndarray,
    train_indices: np.ndarray,
    cfg: Config,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:
    counts = np.bincount(
        labels[train_indices],
        minlength=cfg.regimes,
    ).astype(float)

    proportions = counts / counts.sum()

    class_weights = (
        counts.sum()
        / (cfg.regimes * np.maximum(counts, 1.0))
    )
    class_weights[1] *= cfg.medium_class_multiplier
    class_weights /= class_weights.mean()

    transitions = np.ones(
        (len(cfg.tickers), cfg.regimes, cfg.regimes),
        dtype=float,
    )
    initial = np.ones(
        (len(cfg.tickers), cfg.regimes),
        dtype=float,
    )

    training_set = set(train_indices.tolist())

    for ticker_id in range(len(cfg.tickers)):
        ordered_indices = (
            metadata[
                metadata["ticker_id"] == ticker_id
            ]
            .sort_values("origin_date")
            .index
            .to_numpy(dtype=int)
        )
        ordered_indices = np.array([
            index
            for index in ordered_indices
            if index in training_set
        ], dtype=int)

        if len(ordered_indices) == 0:
            continue

        initial[
            ticker_id,
            labels[ordered_indices[0]],
        ] += 1.0

        for previous, current in zip(
            labels[ordered_indices[:-1]],
            labels[ordered_indices[1:]],
        ):
            transitions[
                ticker_id,
                previous,
                current,
            ] += 1.0

    transitions /= transitions.sum(
        axis=2,
        keepdims=True,
    )
    initial /= initial.sum(axis=1, keepdims=True)

    return (
        class_weights.astype(np.float32),
        proportions.astype(np.float32),
        transitions.astype(np.float32),
        initial.astype(np.float32),
    )


def assert_split_integrity(
    dataset: MarketDataset,
    train_indices: np.ndarray,
    validation_indices: np.ndarray,
    calibration_indices: np.ndarray,
    test_indices: np.ndarray,
    train_cutoff: pd.Timestamp,
    validation_cutoff: pd.Timestamp,
    calibration_cutoff: pd.Timestamp,
) -> pd.DataFrame:
    metadata = dataset.metadata
    sections = {
        "train": train_indices,
        "validation": validation_indices,
        "calibration": calibration_indices,
        "test": test_indices,
    }

    all_indices = np.concatenate(list(sections.values()))
    if len(np.unique(all_indices)) != len(all_indices):
        raise AssertionError("Split index sets overlap.")

    checks = []

    train_target_max = pd.to_datetime(
        metadata.iloc[train_indices]["target_end_date"]
    ).max()
    validation_origin_min = pd.to_datetime(
        metadata.iloc[validation_indices]["origin_date"]
    ).min()
    validation_target_max = pd.to_datetime(
        metadata.iloc[validation_indices]["target_end_date"]
    ).max()
    calibration_origin_min = pd.to_datetime(
        metadata.iloc[calibration_indices]["origin_date"]
    ).min()
    calibration_target_max = pd.to_datetime(
        metadata.iloc[calibration_indices]["target_end_date"]
    ).max()
    test_origin_min = pd.to_datetime(
        metadata.iloc[test_indices]["origin_date"]
    ).min()

    conditions = [
        (
            "training targets end by training cutoff",
            train_target_max <= train_cutoff,
            train_target_max,
            train_cutoff,
        ),
        (
            "validation origins begin after training cutoff",
            validation_origin_min > train_cutoff,
            validation_origin_min,
            train_cutoff,
        ),
        (
            "validation targets end by validation cutoff",
            validation_target_max <= validation_cutoff,
            validation_target_max,
            validation_cutoff,
        ),
        (
            "calibration origins begin after validation cutoff",
            calibration_origin_min > validation_cutoff,
            calibration_origin_min,
            validation_cutoff,
        ),
        (
            "calibration targets end by calibration cutoff",
            calibration_target_max <= calibration_cutoff,
            calibration_target_max,
            calibration_cutoff,
        ),
        (
            "test origins begin after calibration cutoff",
            test_origin_min > calibration_cutoff,
            test_origin_min,
            calibration_cutoff,
        ),
    ]

    for description, passed, observed, boundary in conditions:
        checks.append({
            "check": description,
            "passed": bool(passed),
            "observed": observed,
            "boundary": boundary,
        })
        if not passed:
            raise AssertionError(description)

    return pd.DataFrame(checks)


def prepare_split(
    dataset: MarketDataset,
    cfg: Config,
) -> PreparedSplit:
    metadata = dataset.metadata
    unique_dates = np.array(
        sorted(
            pd.to_datetime(
                metadata["origin_date"]
            ).unique()
        )
    )

    train_cutoff = cutoff_from_fraction(
        unique_dates,
        cfg.train_fraction,
    )
    validation_cutoff = cutoff_from_fraction(
        unique_dates,
        cfg.train_fraction + cfg.validation_fraction,
    )
    calibration_cutoff = cutoff_from_fraction(
        unique_dates,
        cfg.train_fraction
        + cfg.validation_fraction
        + cfg.calibration_fraction,
    )

    origin = pd.to_datetime(metadata["origin_date"])
    target_end = pd.to_datetime(metadata["target_end_date"])

    train_indices = np.flatnonzero(
        (target_end <= train_cutoff).to_numpy()
    )
    validation_indices = np.flatnonzero(
        (
            (origin > train_cutoff)
            & (target_end <= validation_cutoff)
        ).to_numpy()
    )
    calibration_indices = np.flatnonzero(
        (
            (origin > validation_cutoff)
            & (target_end <= calibration_cutoff)
        ).to_numpy()
    )
    test_indices = np.flatnonzero(
        (origin > calibration_cutoff).to_numpy()
    )

    if min(
        len(train_indices),
        len(validation_indices),
        len(calibration_indices),
        len(test_indices),
    ) == 0:
        raise ValueError(
            "At least one chronological section is empty."
        )

    leakage_audit = assert_split_integrity(
        dataset,
        train_indices,
        validation_indices,
        calibration_indices,
        test_indices,
        train_cutoff,
        validation_cutoff,
        calibration_cutoff,
    )

    standardiser = RobustStandardiser.fit(
        dataset.features[train_indices],
        cfg.winsor_lower_quantile,
        cfg.winsor_upper_quantile,
        cfg.standardised_clip,
    )

    scaled_features = standardiser.transform(
        dataset.features
    )
    ticker_one_hot = np.eye(
        len(cfg.tickers),
        dtype=np.float32,
    )[
        metadata["ticker_id"].to_numpy(dtype=int)
    ]
    context = np.concatenate(
        [scaled_features, ticker_one_hot],
        axis=1,
    ).astype(np.float32)

    ticker_thresholds, labels = (
        build_ticker_thresholds_and_labels(
            dataset,
            train_indices,
            cfg,
        )
    )

    (
        class_weights,
        train_proportions,
        transition_matrices,
        initial_probabilities,
    ) = build_training_regime_statistics(
        metadata,
        labels,
        train_indices,
        cfg,
    )

    metadata.loc[:, "split"] = "unused"
    metadata.loc[train_indices, "split"] = "train"
    metadata.loc[validation_indices, "split"] = "validation"
    metadata.loc[calibration_indices, "split"] = "calibration"
    metadata.loc[test_indices, "split"] = "test"
    metadata.loc[:, "regime"] = labels
    metadata.loc[:, "regime_name"] = REGIME_NAMES[labels]

    print("\nStrict chronological sections:")
    print(f"  Train:       {len(train_indices):,}")
    print(f"  Validation:  {len(validation_indices):,}")
    print(f"  Calibration: {len(calibration_indices):,}")
    print(f"  Test:        {len(test_indices):,}")
    print(f"  Train cutoff:       {train_cutoff.date()}")
    print(f"  Validation cutoff:  {validation_cutoff.date()}")
    print(f"  Calibration cutoff: {calibration_cutoff.date()}")

    print("\nPer-ticker training regime boundaries:")
    for ticker_id, ticker in enumerate(cfg.tickers):
        print(
            f"  {ticker}: "
            f"{math.exp(ticker_thresholds[ticker_id, 0]):.3f}, "
            f"{math.exp(ticker_thresholds[ticker_id, 1]):.3f}"
        )

    print("\nLeakage audit:")
    print(leakage_audit.to_string(index=False))

    return PreparedSplit(
        train_indices=train_indices,
        validation_indices=validation_indices,
        calibration_indices=calibration_indices,
        test_indices=test_indices,
        context=context,
        regime_labels=labels,
        ticker_thresholds=ticker_thresholds,
        class_weights=class_weights,
        train_regime_proportions=train_proportions,
        transition_matrices=transition_matrices,
        initial_regime_probabilities=initial_probabilities,
        standardiser=standardiser,
        train_cutoff=train_cutoff,
        validation_cutoff=validation_cutoff,
        calibration_cutoff=calibration_cutoff,
        leakage_audit=leakage_audit,
    )

## Single-pass conditional flow mixture

In [ ]:
def stable_log_cosh(value: torch.Tensor) -> torch.Tensor:
    return (
        torch.logaddexp(value, -value)
        - math.log(2.0)
    )


@dataclass
class ForwardBundle:
    logits: torch.Tensor
    probabilities: torch.Tensor
    locations: torch.Tensor
    scales: torch.Tensor
    skews: torch.Tensor
    tails: torch.Tensor
    expert_medians: torch.Tensor
    mixture_median_approximation: torch.Tensor
    quantile_lower: torch.Tensor
    quantile_median: torch.Tensor
    quantile_upper: torch.Tensor
    component_log_probabilities: torch.Tensor | None
    mixture_log_probability: torch.Tensor | None


class RegimeFlowMixture(nn.Module):
    def __init__(
        self,
        context_dimension: int,
        cfg: Config,
    ):
        super().__init__()
        self.cfg = cfg

        self.encoder = nn.Sequential(
            nn.Linear(
                context_dimension,
                cfg.hidden_size,
            ),
            nn.LayerNorm(cfg.hidden_size),
            nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(
                cfg.hidden_size,
                cfg.hidden_size,
            ),
            nn.SiLU(),
        )

        self.gate_head = nn.Linear(
            cfg.hidden_size,
            cfg.regimes,
        )
        self.quantile_head = nn.Linear(
            cfg.hidden_size,
            3,
        )
        self.expert_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(
                    cfg.hidden_size,
                    cfg.hidden_size,
                ),
                nn.SiLU(),
                nn.Linear(cfg.hidden_size, 4),
            )
            for _ in range(cfg.regimes)
        ])

        for head in self.expert_heads:
            final_layer = head[-1]
            nn.init.zeros_(final_layer.weight)
            with torch.no_grad():
                final_layer.bias[:] = torch.tensor([
                    -1.8,
                    0.0,
                    0.0,
                    0.0,
                ])

    def forward_bundle(
        self,
        context: torch.Tensor,
        target: torch.Tensor | None = None,
    ) -> ForwardBundle:
        # Exactly one stochastic encoder pass per batch.
        hidden = self.encoder(context)

        logits = self.gate_head(hidden)
        probabilities = F.softmax(logits, dim=1)

        raw_quantiles = self.quantile_head(hidden)
        quantile_median = raw_quantiles[:, 0]
        quantile_lower = (
            quantile_median
            - F.softplus(raw_quantiles[:, 1])
        )
        quantile_upper = (
            quantile_median
            + F.softplus(raw_quantiles[:, 2])
        )

        raw_parameters = torch.stack(
            [head(hidden) for head in self.expert_heads],
            dim=1,
        )

        locations = raw_parameters[:, :, 0]
        scales = (
            self.cfg.minimum_scale
            + (
                self.cfg.maximum_scale
                - self.cfg.minimum_scale
            )
            * torch.sigmoid(raw_parameters[:, :, 1])
        )
        skews = (
            self.cfg.maximum_skew
            * torch.tanh(raw_parameters[:, :, 2])
        )
        tails = (
            self.cfg.minimum_tail
            + (
                self.cfg.maximum_tail
                - self.cfg.minimum_tail
            )
            * torch.sigmoid(raw_parameters[:, :, 3])
        )

        expert_medians = (
            locations
            + scales * torch.sinh(skews / tails)
        )
        mixture_median_approximation = torch.sum(
            probabilities * expert_medians,
            dim=1,
        )

        component_log_probabilities = None
        mixture_log_probability = None

        if target is not None:
            target_values = target.squeeze(-1).unsqueeze(1)
            standardised = (
                target_values - locations
            ) / scales

            inverse_argument = (
                tails * torch.asinh(standardised)
                - skews
            )
            inverse_argument = torch.clamp(
                inverse_argument,
                -12.0,
                12.0,
            )
            base = torch.sinh(inverse_argument)

            base_log_density = (
                -0.5 * base**2
                - 0.5 * math.log(2.0 * math.pi)
            )
            log_jacobian = (
                torch.log(tails)
                - torch.log(scales)
                + stable_log_cosh(inverse_argument)
                - 0.5 * torch.log1p(standardised**2)
            )

            component_log_probabilities = (
                base_log_density + log_jacobian
            )
            mixture_log_probability = torch.logsumexp(
                F.log_softmax(logits, dim=1)
                + component_log_probabilities,
                dim=1,
            )

        return ForwardBundle(
            logits=logits,
            probabilities=probabilities,
            locations=locations,
            scales=scales,
            skews=skews,
            tails=tails,
            expert_medians=expert_medians,
            mixture_median_approximation=(
                mixture_median_approximation
            ),
            quantile_lower=quantile_lower,
            quantile_median=quantile_median,
            quantile_upper=quantile_upper,
            component_log_probabilities=(
                component_log_probabilities
            ),
            mixture_log_probability=mixture_log_probability,
        )

    def sample_from_bundle(
        self,
        bundle: ForwardBundle,
        number_of_samples: int,
    ) -> torch.Tensor:
        batch_size = bundle.locations.shape[0]

        noise = torch.randn(
            batch_size,
            self.cfg.regimes,
            number_of_samples,
            device=bundle.locations.device,
            dtype=bundle.locations.dtype,
        )

        transformed = torch.sinh(
            (
                torch.asinh(noise)
                + bundle.skews.unsqueeze(-1)
            )
            / bundle.tails.unsqueeze(-1)
        )

        component_samples = (
            bundle.locations.unsqueeze(-1)
            + bundle.scales.unsqueeze(-1)
            * transformed
        )

        selected_components = torch.distributions.Categorical(
            probs=bundle.probabilities
        ).sample((number_of_samples,)).transpose(0, 1)

        selected = torch.gather(
            component_samples.permute(0, 2, 1),
            dim=2,
            index=selected_components.unsqueeze(-1),
        ).squeeze(-1)

        return selected

## Consistent training objective and early stopping

In [ ]:
def quantile_loss(
    prediction: torch.Tensor,
    target: torch.Tensor,
    quantile: float,
) -> torch.Tensor:
    error = target - prediction
    return torch.maximum(
        quantile * error,
        (quantile - 1.0) * error,
    ).mean()


def objective(
    model: RegimeFlowMixture,
    context: torch.Tensor,
    target: torch.Tensor,
    labels: torch.Tensor,
    class_weights: torch.Tensor,
    target_gate_usage: torch.Tensor,
    cfg: Config,
) -> tuple[torch.Tensor, dict[str, float]]:
    bundle = model.forward_bundle(context, target)

    if (
        bundle.mixture_log_probability is None
        or bundle.component_log_probabilities is None
    ):
        raise RuntimeError("Training bundle lacks log probabilities.")

    mixture_nll = -bundle.mixture_log_probability.mean()

    selected_expert_nll = (
        -bundle.component_log_probabilities.gather(
            1,
            labels.unsqueeze(1),
        ).mean()
    )

    classification = F.cross_entropy(
        bundle.logits,
        labels,
        weight=class_weights,
        label_smoothing=cfg.label_smoothing,
    )

    average_gate_usage = bundle.probabilities.mean(dim=0)
    gate_balance = torch.mean(
        (
            average_gate_usage
            - target_gate_usage
        ) ** 2
    )

    actual_log_volatility = target.squeeze(1)
    predicted_variance = torch.exp(
        2.0 * bundle.mixture_median_approximation
    ).clamp_min(EPS)
    actual_variance = torch.exp(
        2.0 * actual_log_volatility
    ).clamp_min(EPS)

    variance_ratio = (
        actual_variance / predicted_variance
    )
    qlike = torch.mean(
        variance_ratio
        - torch.log(variance_ratio)
        - 1.0
    )

    quantile_objective = (
        quantile_loss(
            bundle.quantile_lower,
            actual_log_volatility,
            0.05,
        )
        + quantile_loss(
            bundle.quantile_median,
            actual_log_volatility,
            0.50,
        )
        + quantile_loss(
            bundle.quantile_upper,
            actual_log_volatility,
            0.95,
        )
    )

    total = (
        mixture_nll
        + cfg.expert_alignment_weight
        * selected_expert_nll
        + cfg.regime_classification_weight
        * classification
        + cfg.gate_balance_weight
        * gate_balance
        + cfg.qlike_weight * qlike
        + cfg.quantile_weight
        * quantile_objective
    )

    metrics = {
        "total": float(total.detach().cpu()),
        "nll": float(mixture_nll.detach().cpu()),
        "classification": float(
            classification.detach().cpu()
        ),
        "qlike": float(qlike.detach().cpu()),
        "accuracy": float(
            (
                bundle.logits.argmax(dim=1)
                == labels
            )
            .float()
            .mean()
            .detach()
            .cpu()
        ),
        "medium_probability": float(
            bundle.probabilities[:, 1]
            .mean()
            .detach()
            .cpu()
        ),
    }

    return total, metrics


def make_loader(
    split: PreparedSplit,
    dataset: MarketDataset,
    indices: np.ndarray,
    cfg: Config,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    data = TensorDataset(
        torch.from_numpy(
            split.context[indices]
        ).float(),
        torch.from_numpy(
            dataset.targets_log_vol[indices]
        ).float().unsqueeze(1),
        torch.from_numpy(
            split.regime_labels[indices]
        ).long(),
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        data,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        drop_last=False,
        generator=generator,
        num_workers=0,
    )


def run_epoch(
    model: RegimeFlowMixture,
    loader: DataLoader,
    cfg: Config,
    class_weights: torch.Tensor,
    target_gate_usage: torch.Tensor,
    optimiser: optim.Optimizer | None,
) -> dict[str, float]:
    training = optimiser is not None
    model.train(training)

    totals = {}
    observation_count = 0

    for context, target, labels in loader:
        context = context.to(DEVICE)
        target = target.to(DEVICE)
        labels = labels.to(DEVICE)
        batch_size = len(context)

        if training:
            optimiser.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            total, metrics = objective(
                model,
                context,
                target,
                labels,
                class_weights,
                target_gate_usage,
                cfg,
            )

            if training:
                total.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    cfg.gradient_clip,
                )
                optimiser.step()

        for key, value in metrics.items():
            totals[key] = (
                totals.get(key, 0.0)
                + value * batch_size
            )
        observation_count += batch_size

    return {
        key: value / observation_count
        for key, value in totals.items()
    }


def fit_single_model(
    dataset: MarketDataset,
    split: PreparedSplit,
    cfg: Config,
    seed: int,
) -> tuple[
    RegimeFlowMixture,
    dict[str, list[float]],
    int,
]:
    set_seed(seed)

    train_loader = make_loader(
        split,
        dataset,
        split.train_indices,
        cfg,
        shuffle=True,
        seed=seed,
    )
    validation_loader = make_loader(
        split,
        dataset,
        split.validation_indices,
        cfg,
        shuffle=False,
        seed=seed,
    )

    model = RegimeFlowMixture(
        split.context.shape[1],
        cfg,
    ).to(DEVICE)

    optimiser = optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimiser,
        mode="min",
        factor=0.5,
        patience=max(2, cfg.patience // 2),
    )

    class_weights = torch.from_numpy(
        split.class_weights
    ).float().to(DEVICE)
    target_gate_usage = torch.from_numpy(
        split.train_regime_proportions
    ).float().to(DEVICE)

    history = {
        "train_loss": [],
        "validation_loss": [],
        "validation_nll": [],
        "validation_accuracy": [],
        "validation_medium_probability": [],
    }

    best_validation_loss = float("inf")
    best_state = None
    best_epoch = 0
    stale_epochs = 0

    print(f"\nTraining seed {seed}...")

    for epoch in range(1, cfg.epochs + 1):
        train_metrics = run_epoch(
            model,
            train_loader,
            cfg,
            class_weights,
            target_gate_usage,
            optimiser,
        )

        with torch.no_grad():
            validation_metrics = run_epoch(
                model,
                validation_loader,
                cfg,
                class_weights,
                target_gate_usage,
                optimiser=None,
            )

        scheduler.step(validation_metrics["total"])

        history["train_loss"].append(
            train_metrics["total"]
        )
        history["validation_loss"].append(
            validation_metrics["total"]
        )
        history["validation_nll"].append(
            validation_metrics["nll"]
        )
        history["validation_accuracy"].append(
            validation_metrics["accuracy"]
        )
        history[
            "validation_medium_probability"
        ].append(
            validation_metrics["medium_probability"]
        )

        if (
            validation_metrics["total"]
            < best_validation_loss - 1e-4
        ):
            best_validation_loss = (
                validation_metrics["total"]
            )
            best_state = copy.deepcopy(
                model.state_dict()
            )
            best_epoch = epoch
            stale_epochs = 0
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % cfg.print_every == 0:
            print(
                f"  Epoch {epoch:03d}/{cfg.epochs} | "
                f"train={train_metrics['total']:.4f} | "
                f"validation={validation_metrics['total']:.4f} | "
                f"accuracy={validation_metrics['accuracy']:.3f} | "
                f"P(medium)={validation_metrics['medium_probability']:.3f}"
            )

        if stale_epochs >= cfg.patience:
            print(
                f"  Early stopping; best epoch={best_epoch}"
            )
            break

    if best_state is None:
        raise RuntimeError(
            "Training failed to create a checkpoint."
        )

    model.load_state_dict(best_state)
    model.eval()

    return model, history, best_epoch


def fit_ensemble(
    dataset: MarketDataset,
    split: PreparedSplit,
    cfg: Config,
) -> tuple[
    list[RegimeFlowMixture],
    list[dict[str, list[float]]],
    list[int],
]:
    models = []
    histories = []
    best_epochs = []

    for seed in cfg.ensemble_seeds:
        model, history, best_epoch = fit_single_model(
            dataset,
            split,
            cfg,
            seed,
        )
        models.append(model)
        histories.append(history)
        best_epochs.append(best_epoch)

    return models, histories, best_epochs

## Fair ticker-specific HAR baseline and ensemble outputs

In [ ]:
def fit_ticker_specific_har(
    dataset: MarketDataset,
    split: PreparedSplit,
    cfg: Config,
) -> np.ndarray:
    feature_columns = [
        dataset.feature_names.index(
            "log_realised_volatility_5"
        ),
        dataset.feature_names.index(
            "log_realised_volatility_20"
        ),
        dataset.feature_names.index(
            "log_realised_volatility_60"
        ),
    ]

    predictions = np.full(
        len(dataset.metadata),
        np.nan,
        dtype=float,
    )

    for ticker_id, ticker in enumerate(cfg.tickers):
        ticker_train = split.train_indices[
            dataset.metadata.iloc[
                split.train_indices
            ]["ticker_id"].to_numpy(dtype=int)
            == ticker_id
        ]
        ticker_all = np.flatnonzero(
            dataset.metadata[
                "ticker_id"
            ].to_numpy(dtype=int)
            == ticker_id
        )

        training_design = dataset.features[
            ticker_train
        ][:, feature_columns].astype(float)
        training_design = np.column_stack([
            np.ones(len(training_design)),
            training_design,
        ])
        training_target = (
            dataset.targets_log_vol[ticker_train]
        )

        penalty = 1e-4 * np.eye(
            training_design.shape[1]
        )
        penalty[0, 0] = 0.0

        coefficients = np.linalg.solve(
            training_design.T @ training_design
            + penalty,
            training_design.T @ training_target,
        )

        all_design = dataset.features[
            ticker_all
        ][:, feature_columns].astype(float)
        all_design = np.column_stack([
            np.ones(len(all_design)),
            all_design,
        ])

        predictions[ticker_all] = np.exp(
            all_design @ coefficients
        )

    if not np.all(np.isfinite(predictions)):
        raise ValueError(
            "Ticker-specific HAR predictions are incomplete."
        )

    return predictions


def softmax_numpy(
    logits: np.ndarray,
    temperature: float,
) -> np.ndarray:
    scaled = logits / temperature
    scaled = scaled - np.max(
        scaled,
        axis=-1,
        keepdims=True,
    )
    exponentials = np.exp(scaled)
    return exponentials / exponentials.sum(
        axis=-1,
        keepdims=True,
    )


@torch.no_grad()
def collect_ensemble_outputs(
    models: list[RegimeFlowMixture],
    dataset: MarketDataset,
    split: PreparedSplit,
    indices: np.ndarray,
    cfg: Config,
) -> dict[str, np.ndarray]:
    samples_by_model = []
    logits_by_model = []
    log_probabilities_by_model = []

    samples_per_model = max(
        32,
        math.ceil(
            cfg.prediction_samples / len(models)
        ),
    )

    for model in models:
        model_samples = []
        model_logits = []
        model_log_probabilities = []

        for start in range(
            0,
            len(indices),
            cfg.prediction_batch_size,
        ):
            batch_indices = indices[
                start:start
                + cfg.prediction_batch_size
            ]

            context = torch.from_numpy(
                split.context[batch_indices]
            ).float().to(DEVICE)
            target = torch.from_numpy(
                dataset.targets_log_vol[
                    batch_indices
                ]
            ).float().unsqueeze(1).to(DEVICE)

            bundle = model.forward_bundle(
                context,
                target,
            )
            log_samples = model.sample_from_bundle(
                bundle,
                samples_per_model,
            )

            if bundle.mixture_log_probability is None:
                raise RuntimeError(
                    "Evaluation bundle lacks log probability."
                )

            model_samples.append(
                log_samples.cpu().numpy()
            )
            model_logits.append(
                bundle.logits.cpu().numpy()
            )
            model_log_probabilities.append(
                bundle.mixture_log_probability
                .cpu()
                .numpy()
            )

        samples_by_model.append(
            np.concatenate(model_samples, axis=0)
        )
        logits_by_model.append(
            np.concatenate(model_logits, axis=0)
        )
        log_probabilities_by_model.append(
            np.concatenate(
                model_log_probabilities,
                axis=0,
            )
        )

    log_samples = np.concatenate(
        samples_by_model,
        axis=1,
    )
    logits = np.stack(
        logits_by_model,
        axis=0,
    )

    stacked_log_probabilities = np.stack(
        log_probabilities_by_model,
        axis=1,
    )
    maximum = np.max(
        stacked_log_probabilities,
        axis=1,
        keepdims=True,
    )
    ensemble_log_probability = (
        maximum[:, 0]
        + np.log(
            np.mean(
                np.exp(
                    stacked_log_probabilities - maximum
                ),
                axis=1,
            )
            + EPS
        )
    )

    return {
        "log_samples": log_samples,
        "model_logits": logits,
        "uncalibrated_log_probability": (
            ensemble_log_probability
        ),
    }

## Calibration-only interval scale and regime temperature

In [ ]:
def interval_score(
    lower: np.ndarray,
    upper: np.ndarray,
    actual: np.ndarray,
    alpha: float,
) -> np.ndarray:
    score = upper - lower
    score += (
        2.0 / alpha
        * (lower - actual)
        * (actual < lower)
    )
    score += (
        2.0 / alpha
        * (actual - upper)
        * (actual > upper)
    )
    return score


def calibrate_interval_scale(
    log_samples: np.ndarray,
    actual_log_volatility: np.ndarray,
    cfg: Config,
) -> float:
    median = np.median(
        log_samples,
        axis=1,
        keepdims=True,
    )
    actual = np.exp(actual_log_volatility)

    candidates = np.linspace(
        cfg.interval_scale_min,
        cfg.interval_scale_max,
        cfg.interval_scale_candidates,
    )

    best_scale = 1.0
    best_score = float("inf")

    for scale in candidates:
        adjusted = (
            median
            + scale * (log_samples - median)
        )
        adjusted = np.clip(
            adjusted,
            math.log(cfg.minimum_reported_volatility),
            math.log(cfg.maximum_reported_volatility),
        )
        samples = np.exp(adjusted)

        q05, q25, q75, q95 = np.quantile(
            samples,
            [0.05, 0.25, 0.75, 0.95],
            axis=1,
        )

        score = np.mean(
            interval_score(
                q05,
                q95,
                actual,
                alpha=0.10,
            )
            + interval_score(
                q25,
                q75,
                actual,
                alpha=0.50,
            )
        )

        if score < best_score:
            best_score = score
            best_scale = float(scale)

    print(
        f"Calibration interval scale: {best_scale:.3f}"
    )
    return best_scale


def multiclass_log_loss(
    probabilities: np.ndarray,
    labels: np.ndarray,
) -> float:
    selected = probabilities[
        np.arange(len(labels)),
        labels,
    ]
    return float(
        -np.mean(np.log(selected + EPS))
    )


def calibrate_regime_temperature(
    model_logits: np.ndarray,
    labels: np.ndarray,
    cfg: Config,
) -> float:
    candidates = np.linspace(
        cfg.temperature_min,
        cfg.temperature_max,
        cfg.temperature_candidates,
    )

    best_temperature = 1.0
    best_loss = float("inf")

    for temperature in candidates:
        probabilities = np.mean(
            softmax_numpy(
                model_logits,
                temperature,
            ),
            axis=0,
        )
        loss = multiclass_log_loss(
            probabilities,
            labels,
        )

        if loss < best_loss:
            best_loss = loss
            best_temperature = float(temperature)

    print(
        f"Calibration regime temperature: "
        f"{best_temperature:.3f}"
    )
    return best_temperature


def calibrated_probabilities(
    model_logits: np.ndarray,
    temperature: float,
) -> np.ndarray:
    return np.mean(
        softmax_numpy(
            model_logits,
            temperature,
        ),
        axis=0,
    )

## Predictions and past-only regime smoothing

In [ ]:
def empirical_crps_numpy(
    samples: np.ndarray,
    actual: np.ndarray,
) -> np.ndarray:
    first_term = np.mean(
        np.abs(samples - actual[:, None]),
        axis=1,
    )

    sorted_samples = np.sort(samples, axis=1)
    sample_count = samples.shape[1]
    ranks = np.arange(
        1,
        sample_count + 1,
        dtype=float,
    )
    coefficients = (
        2.0 * ranks - sample_count - 1.0
    )
    second_term = (
        sorted_samples
        * coefficients[None, :]
    ).sum(axis=1) / (sample_count**2)

    return first_term - second_term


def apply_regime_smoothing(
    predictions: pd.DataFrame,
    split: PreparedSplit,
) -> pd.DataFrame:
    predictions = predictions.copy()

    smoothed_columns = [
        "smoothed_probability_low",
        "smoothed_probability_medium",
        "smoothed_probability_high",
    ]
    for column in smoothed_columns:
        predictions[column] = 0.0

    for ticker_id, ticker_rows in predictions.groupby(
        "ticker_id",
        sort=False,
    ):
        ordered = ticker_rows.sort_values("origin_date")
        posterior = split.initial_regime_probabilities[
            int(ticker_id)
        ].astype(float).copy()
        transition = split.transition_matrices[
            int(ticker_id)
        ]

        for row_index in ordered.index:
            gate_probability = predictions.loc[
                row_index,
                [
                    "probability_low",
                    "probability_medium",
                    "probability_high",
                ],
            ].to_numpy(dtype=float)

            # Uses only the previous filtered state, the training-only
            # transition matrix, and the current forecast probability.
            prior = posterior @ transition
            posterior = prior * gate_probability
            posterior /= posterior.sum() + EPS

            predictions.loc[
                row_index,
                smoothed_columns,
            ] = posterior

    predictions["smoothed_regime"] = (
        predictions[smoothed_columns]
        .to_numpy()
        .argmax(axis=1)
    )
    predictions["smoothed_regime_name"] = REGIME_NAMES[
        predictions[
            "smoothed_regime"
        ].to_numpy(dtype=int)
    ]

    return predictions


def build_prediction_frame(
    ensemble_outputs: dict[str, np.ndarray],
    dataset: MarketDataset,
    split: PreparedSplit,
    indices: np.ndarray,
    interval_scale: float,
    regime_temperature: float,
    har_predictions: np.ndarray,
    cfg: Config,
) -> pd.DataFrame:
    raw_log_samples = ensemble_outputs["log_samples"]
    median_log = np.median(
        raw_log_samples,
        axis=1,
        keepdims=True,
    )

    calibrated_log_samples = (
        median_log
        + interval_scale
        * (raw_log_samples - median_log)
    )
    calibrated_log_samples = np.clip(
        calibrated_log_samples,
        math.log(cfg.minimum_reported_volatility),
        math.log(cfg.maximum_reported_volatility),
    )
    samples = np.exp(calibrated_log_samples)

    actual = np.exp(
        dataset.targets_log_vol[indices]
    )
    quantiles = np.quantile(
        samples,
        [0.05, 0.25, 0.50, 0.75, 0.95],
        axis=1,
    )

    probabilities = calibrated_probabilities(
        ensemble_outputs["model_logits"],
        regime_temperature,
    )

    frame = (
        dataset.metadata
        .iloc[indices]
        .copy()
        .reset_index(drop=True)
    )

    frame["predicted_mean_vol"] = samples.mean(axis=1)
    frame["predicted_q05_vol"] = quantiles[0]
    frame["predicted_q25_vol"] = quantiles[1]
    frame["predicted_median_vol"] = quantiles[2]
    frame["predicted_q75_vol"] = quantiles[3]
    frame["predicted_q95_vol"] = quantiles[4]

    # This density belongs to the original model, before sample-spread
    # calibration. The name makes that distinction explicit.
    frame[
        "uncalibrated_negative_log_likelihood"
    ] = -ensemble_outputs[
        "uncalibrated_log_probability"
    ]

    frame["crps"] = empirical_crps_numpy(
        samples,
        actual,
    )
    frame["pit"] = np.mean(
        samples <= actual[:, None],
        axis=1,
    )
    frame["har_style_baseline"] = (
        har_predictions[indices]
    )

    for regime_id, regime_name in enumerate(REGIME_NAMES):
        frame[
            f"probability_{regime_name.lower()}"
        ] = probabilities[:, regime_id]

    frame["predicted_regime"] = probabilities.argmax(axis=1)
    frame["predicted_regime_name"] = REGIME_NAMES[
        frame[
            "predicted_regime"
        ].to_numpy(dtype=int)
    ]

    return apply_regime_smoothing(
        frame,
        split,
    )

## Pooled, per-ticker, regime, non-overlapping, and bootstrap evaluation

In [ ]:
MODEL_COLUMNS = {
    "SigFlow v4": "predicted_median_vol",
    "Rolling volatility": "rolling_vol_baseline",
    "EWMA volatility": "ewma_vol_baseline",
    "HAR-style per ticker": "har_style_baseline",
}


def qlike(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> float:
    actual_variance = np.maximum(actual**2, EPS)
    predicted_variance = np.maximum(
        predicted**2,
        EPS,
    )
    ratio = actual_variance / predicted_variance
    return float(
        np.mean(ratio - np.log(ratio) - 1.0)
    )


def point_metrics(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, float]:
    error = predicted - actual

    correlation = (
        float(
            np.corrcoef(actual, predicted)[0, 1]
        )
        if (
            np.std(actual) > EPS
            and np.std(predicted) > EPS
        )
        else float("nan")
    )

    return {
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(
            np.sqrt(np.mean(error**2))
        ),
        "correlation": correlation,
        "qlike": qlike(actual, predicted),
    }


def confusion_matrix_numpy(
    actual: np.ndarray,
    predicted: np.ndarray,
    classes: int,
) -> np.ndarray:
    matrix = np.zeros(
        (classes, classes),
        dtype=int,
    )

    for actual_value, predicted_value in zip(
        actual,
        predicted,
    ):
        matrix[
            int(actual_value),
            int(predicted_value),
        ] += 1

    return matrix


def regime_metrics(
    labels: np.ndarray,
    predictions: np.ndarray,
    probabilities: np.ndarray,
    classes: int,
) -> tuple[dict[str, float], np.ndarray]:
    matrix = confusion_matrix_numpy(
        labels,
        predictions,
        classes,
    )

    recall = np.diag(matrix) / np.maximum(
        matrix.sum(axis=1),
        1,
    )
    precision = np.diag(matrix) / np.maximum(
        matrix.sum(axis=0),
        1,
    )
    f1 = (
        2.0 * precision * recall
        / np.maximum(precision + recall, EPS)
    )

    one_hot = np.eye(classes)[labels]
    brier = float(
        np.mean(
            np.sum(
                (probabilities - one_hot) ** 2,
                axis=1,
            )
        )
    )

    metrics = {
        "accuracy": float(
            np.mean(labels == predictions)
        ),
        "balanced_accuracy": float(
            np.mean(recall)
        ),
        "macro_f1": float(np.mean(f1)),
        "recall_low": float(recall[0]),
        "recall_medium": float(recall[1]),
        "recall_high": float(recall[2]),
        "brier_score": brier,
        "log_loss": multiclass_log_loss(
            probabilities,
            labels,
        ),
    }

    return metrics, matrix


def evaluate_point_models(
    predictions: pd.DataFrame,
    group_label: str,
) -> pd.DataFrame:
    rows = []
    actual = predictions["actual_vol"].to_numpy(dtype=float)

    for model_name, column in MODEL_COLUMNS.items():
        row = {
            "group": group_label,
            "model": model_name,
            "observations": len(predictions),
        }
        row.update(
            point_metrics(
                actual,
                predictions[column].to_numpy(dtype=float),
            )
        )
        rows.append(row)

    return pd.DataFrame(rows)


def evaluate_all_groups(
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    tables = [
        evaluate_point_models(
            predictions,
            "Pooled",
        )
    ]

    for ticker, ticker_rows in predictions.groupby(
        "ticker",
        sort=True,
    ):
        tables.append(
            evaluate_point_models(
                ticker_rows,
                ticker,
            )
        )

    return pd.concat(
        tables,
        ignore_index=True,
    )


def non_overlapping_phase_metrics(
    predictions: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    tables = []

    for phase in range(cfg.horizon):
        selected_pieces = []

        for _, ticker_rows in predictions.groupby(
            "ticker",
            sort=False,
        ):
            ordered = ticker_rows.sort_values(
                "origin_date"
            )
            selected_pieces.append(
                ordered.iloc[phase::cfg.horizon]
            )

        selected = pd.concat(
            selected_pieces,
            ignore_index=True,
        )
        table = evaluate_point_models(
            selected,
            group_label=f"phase_{phase}",
        )
        table["phase"] = phase
        tables.append(table)

    return pd.concat(
        tables,
        ignore_index=True,
    )


def paired_block_bootstrap(
    predictions: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    unique_dates = np.array(
        sorted(
            pd.to_datetime(
                predictions["origin_date"]
            ).unique()
        )
    )
    block_length = min(
        cfg.bootstrap_block_days,
        len(unique_dates),
    )
    possible_starts = np.arange(
        max(
            1,
            len(unique_dates) - block_length + 1,
        )
    )

    rng = np.random.default_rng(
        cfg.seed + 12345
    )
    baseline_names = [
        name
        for name in MODEL_COLUMNS
        if name != "SigFlow v4"
    ]

    results = {
        (baseline, metric): []
        for baseline in baseline_names
        for metric in ["mae", "qlike"]
    }

    for _ in range(cfg.bootstrap_repetitions):
        sampled_dates = []

        while len(sampled_dates) < len(unique_dates):
            start = int(rng.choice(possible_starts))
            sampled_dates.extend(
                unique_dates[
                    start:start + block_length
                ].tolist()
            )

        sampled_dates = sampled_dates[
            :len(unique_dates)
        ]

        pieces = [
            predictions[
                pd.to_datetime(
                    predictions["origin_date"]
                )
                == date
            ]
            for date in sampled_dates
        ]
        sample = pd.concat(
            pieces,
            ignore_index=True,
        )

        actual = sample[
            "actual_vol"
        ].to_numpy(dtype=float)
        sigflow = point_metrics(
            actual,
            sample[
                MODEL_COLUMNS["SigFlow v4"]
            ].to_numpy(dtype=float),
        )

        for baseline in baseline_names:
            baseline_metrics = point_metrics(
                actual,
                sample[
                    MODEL_COLUMNS[baseline]
                ].to_numpy(dtype=float),
            )

            for metric in ["mae", "qlike"]:
                # Negative means SigFlow is better.
                results[(baseline, metric)].append(
                    sigflow[metric]
                    - baseline_metrics[metric]
                )

    rows = []
    for (baseline, metric), values in results.items():
        rows.append({
            "comparison": f"SigFlow v4 minus {baseline}",
            "metric": metric,
            "mean_difference": np.mean(values),
            "lower_95": np.quantile(values, 0.025),
            "upper_95": np.quantile(values, 0.975),
            "sigflow_better_if": "difference < 0",
        })

    return pd.DataFrame(rows)


def evaluate_predictions(
    predictions: pd.DataFrame,
    cfg: Config,
) -> dict[str, object]:
    point_table = evaluate_all_groups(predictions)

    actual = predictions["actual_vol"].to_numpy(dtype=float)
    model_summary = {
        "uncalibrated_nll": float(
            predictions[
                "uncalibrated_negative_log_likelihood"
            ].mean()
        ),
        "crps": float(predictions["crps"].mean()),
        "coverage_50": float(np.mean(
            (
                actual
                >= predictions[
                    "predicted_q25_vol"
                ].to_numpy()
            )
            & (
                actual
                <= predictions[
                    "predicted_q75_vol"
                ].to_numpy()
            )
        )),
        "coverage_90": float(np.mean(
            (
                actual
                >= predictions[
                    "predicted_q05_vol"
                ].to_numpy()
            )
            & (
                actual
                <= predictions[
                    "predicted_q95_vol"
                ].to_numpy()
            )
        )),
        "interval_width_90": float(np.mean(
            predictions["predicted_q95_vol"]
            - predictions["predicted_q05_vol"]
        )),
    }

    labels = predictions[
        "regime"
    ].to_numpy(dtype=int)

    raw_probabilities = predictions[
        [
            "probability_low",
            "probability_medium",
            "probability_high",
        ]
    ].to_numpy(dtype=float)
    raw_predictions = predictions[
        "predicted_regime"
    ].to_numpy(dtype=int)

    smoothed_probabilities = predictions[
        [
            "smoothed_probability_low",
            "smoothed_probability_medium",
            "smoothed_probability_high",
        ]
    ].to_numpy(dtype=float)
    smoothed_predictions = predictions[
        "smoothed_regime"
    ].to_numpy(dtype=int)

    raw_regime_metrics, raw_matrix = regime_metrics(
        labels,
        raw_predictions,
        raw_probabilities,
        cfg.regimes,
    )
    smoothed_regime_metrics, smoothed_matrix = (
        regime_metrics(
            labels,
            smoothed_predictions,
            smoothed_probabilities,
            cfg.regimes,
        )
    )

    non_overlapping = non_overlapping_phase_metrics(
        predictions,
        cfg,
    )

    bootstrap = (
        paired_block_bootstrap(
            predictions,
            cfg,
        )
        if cfg.run_block_bootstrap
        else None
    )

    return {
        "point_metrics": point_table,
        "probabilistic_summary": pd.DataFrame([
            model_summary
        ]),
        "raw_regime_metrics": pd.DataFrame([
            raw_regime_metrics
        ]),
        "smoothed_regime_metrics": pd.DataFrame([
            smoothed_regime_metrics
        ]),
        "raw_confusion_matrix": raw_matrix,
        "smoothed_confusion_matrix": smoothed_matrix,
        "non_overlapping_phase_metrics": non_overlapping,
        "paired_bootstrap": bootstrap,
    }

## Graphs, audit files, and checkpoint saving

In [ ]:
def plot_confusion_matrix(
    matrix: np.ndarray,
    title: str,
) -> None:
    plt.figure(figsize=(5, 4))
    plt.imshow(matrix)
    plt.xticks(np.arange(3), REGIME_NAMES)
    plt.yticks(np.arange(3), REGIME_NAMES)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)

    for row in range(3):
        for column in range(3):
            plt.text(
                column,
                row,
                str(matrix[row, column]),
                ha="center",
                va="center",
            )

    plt.colorbar()
    plt.tight_layout()
    plt.show()


def plot_results(
    histories: list[dict[str, list[float]]],
    predictions: pd.DataFrame,
    evaluation: dict[str, object],
    cfg: Config,
) -> None:
    plt.figure(figsize=(10, 4))
    for model_index, history in enumerate(
        histories,
        start=1,
    ):
        epochs = np.arange(
            1,
            len(history["train_loss"]) + 1,
        )
        plt.plot(
            epochs,
            history["train_loss"],
            alpha=0.55,
            label=f"Train {model_index}",
        )
        plt.plot(
            epochs,
            history["validation_loss"],
            label=f"Validation {model_index}",
        )

    plt.xlabel("Epoch")
    plt.ylabel("Objective")
    plt.title("Training and model-validation history")
    plt.legend()
    plt.tight_layout()
    plt.show()

    for ticker in cfg.tickers:
        ticker_data = (
            predictions[
                predictions["ticker"] == ticker
            ]
            .sort_values("origin_date")
            .copy()
        )

        if ticker_data.empty:
            continue

        dates = pd.to_datetime(
            ticker_data["origin_date"]
        )

        plt.figure(figsize=(13, 5))
        plt.plot(
            dates,
            ticker_data["actual_vol"],
            label="Actual future volatility",
        )
        plt.plot(
            dates,
            ticker_data["predicted_median_vol"],
            label="SigFlow v4",
        )
        plt.plot(
            dates,
            ticker_data["ewma_vol_baseline"],
            label="EWMA",
            alpha=0.75,
        )
        plt.plot(
            dates,
            ticker_data["har_style_baseline"],
            label="HAR-style",
            alpha=0.75,
        )
        plt.fill_between(
            dates,
            ticker_data["predicted_q05_vol"],
            ticker_data["predicted_q95_vol"],
            alpha=0.20,
            label="Calibration-tuned 90% interval",
        )
        plt.title(
            f"{ticker}: {cfg.horizon}-day volatility forecast"
        )
        plt.ylabel("Annualised volatility")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(13, 3.5))
        plt.stackplot(
            dates,
            ticker_data[
                "smoothed_probability_low"
            ],
            ticker_data[
                "smoothed_probability_medium"
            ],
            ticker_data[
                "smoothed_probability_high"
            ],
            labels=REGIME_NAMES,
            alpha=0.8,
        )
        plt.ylim(0, 1)
        plt.title(
            f"{ticker}: past-only smoothed regime probabilities"
        )
        plt.ylabel("Probability")
        plt.legend(
            loc="upper left",
            ncol=3,
        )
        plt.tight_layout()
        plt.show()

    plt.figure(figsize=(7, 4))
    plt.hist(
        predictions["pit"],
        bins=np.linspace(0, 1, 11),
    )
    plt.axhline(
        len(predictions) / 10,
        linestyle="--",
    )
    plt.title("Test PIT calibration")
    plt.xlabel("PIT")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    plot_confusion_matrix(
        evaluation["raw_confusion_matrix"],
        "Raw test-regime confusion matrix",
    )
    plot_confusion_matrix(
        evaluation["smoothed_confusion_matrix"],
        "Past-only smoothed test-regime confusion matrix",
    )


def save_results(
    models: list[RegimeFlowMixture],
    histories: list[dict[str, list[float]]],
    predictions: pd.DataFrame,
    evaluation: dict[str, object],
    dataset: MarketDataset,
    split: PreparedSplit,
    interval_scale: float,
    regime_temperature: float,
    cfg: Config,
) -> None:
    output_directory = Path(cfg.output_dir)
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    predictions.to_csv(
        output_directory / "test_predictions.csv",
        index=False,
    )
    dataset.metadata.to_csv(
        output_directory / "all_sample_metadata.csv",
        index=False,
    )
    dataset.data_manifest.to_csv(
        output_directory / "data_manifest.csv",
        index=False,
    )
    dataset.skipped_samples.to_csv(
        output_directory / "skipped_samples.csv",
        index=False,
    )
    split.leakage_audit.to_csv(
        output_directory / "leakage_audit.csv",
        index=False,
    )

    evaluation["point_metrics"].to_csv(
        output_directory / "point_metrics.csv",
        index=False,
    )
    evaluation["probabilistic_summary"].to_csv(
        output_directory
        / "probabilistic_summary.csv",
        index=False,
    )
    evaluation["raw_regime_metrics"].to_csv(
        output_directory / "raw_regime_metrics.csv",
        index=False,
    )
    evaluation["smoothed_regime_metrics"].to_csv(
        output_directory
        / "smoothed_regime_metrics.csv",
        index=False,
    )
    evaluation[
        "non_overlapping_phase_metrics"
    ].to_csv(
        output_directory
        / "non_overlapping_phase_metrics.csv",
        index=False,
    )

    if evaluation["paired_bootstrap"] is not None:
        evaluation["paired_bootstrap"].to_csv(
            output_directory
            / "paired_block_bootstrap.csv",
            index=False,
        )

    for model_index, history in enumerate(
        histories,
        start=1,
    ):
        pd.DataFrame(history).to_csv(
            output_directory
            / f"training_history_model_{model_index}.csv",
            index=False,
        )

    torch.save({
        "model_state_dicts": [
            model.state_dict()
            for model in models
        ],
        "config": asdict(cfg),
        "feature_names": dataset.feature_names,
        "data_source": dataset.data_source,
        "available_market_symbols": (
            dataset.available_market_symbols
        ),
        "winsor_lower": split.standardiser.lower,
        "winsor_upper": split.standardiser.upper,
        "feature_mean": split.standardiser.mean,
        "feature_scale": split.standardiser.scale,
        "ticker_thresholds": split.ticker_thresholds,
        "class_weights": split.class_weights,
        "train_regime_proportions": (
            split.train_regime_proportions
        ),
        "transition_matrices": (
            split.transition_matrices
        ),
        "initial_regime_probabilities": (
            split.initial_regime_probabilities
        ),
        "interval_scale": interval_scale,
        "regime_temperature": regime_temperature,
        "split_cutoffs": {
            "train": str(split.train_cutoff),
            "validation": str(
                split.validation_cutoff
            ),
            "calibration": str(
                split.calibration_cutoff
            ),
        },
    }, output_directory / "sigflow_v4_ensemble.pt")

    with open(
        output_directory / "run_config.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump({
            **asdict(cfg),
            "actual_data_source": dataset.data_source,
            "available_market_symbols": list(
                dataset.available_market_symbols
            ),
            "interval_scale": interval_scale,
            "regime_temperature": regime_temperature,
        }, file, indent=2)

    print(
        "\nSaved outputs to:",
        output_directory.resolve(),
    )

## Test-set discipline

The test section is intentionally untouched until the final evaluation cell.

To preserve that property:

- choose architecture and optimisation settings using training and validation only;
- choose interval scale and regime temperature using calibration only;
- do not repeatedly change the model after looking at test results;
- when a new modelling decision is made from test performance, treat the old test section as development data and create a later holdout period for the next final evaluation.

The notebook saves the exact configuration, cutoffs, leakage audit, preprocessing parameters, calibration settings, and data provenance so each run can be reproduced and compared honestly.

## What “waiting at the start” meant

The earlier notebook allowed each failed network request to wait for 30 seconds and then tried a second provider. With two target stocks and three market-context symbols, a fully blocked runtime could spend several minutes waiting before it reached the fallback.

This version:

- performs one three-second connectivity check;
- uses six-second per-provider timeouts if the check succeeds;
- prints each provider attempt immediately;
- skips directly to synthetic pipeline-test data when quick mode has no internet;
- stops quickly instead of substituting synthetic data in final real-experiment mode.

To bypass the internet completely for a pipeline test, set:

```python
use_real_market_data: bool = False
```

To require genuine prices, retain:

```python
experiment_mode: str = "real_experiment"
allow_synthetic_fallback: bool = False
```

# Run the watertight experiment

In [ ]:
def main(cfg: Config = CFG) -> dict[str, object]:
    print("=" * 78, flush=True)
    print("Starting SigFlow-Sim v4", flush=True)
    print("Mode:", cfg.experiment_mode, flush=True)
    print("=" * 78, flush=True)

    set_seed(cfg.seed)

    dataset = build_market_dataset(cfg)
    split = prepare_split(dataset, cfg)

    har_predictions = fit_ticker_specific_har(
        dataset,
        split,
        cfg,
    )

    models, histories, best_epochs = fit_ensemble(
        dataset,
        split,
        cfg,
    )
    print("\nBest validation epochs:", best_epochs)

    print(
        "\nGenerating calibration forecasts "
        "(not used for model fitting)..."
    )
    calibration_outputs = collect_ensemble_outputs(
        models,
        dataset,
        split,
        split.calibration_indices,
        cfg,
    )

    interval_scale = calibrate_interval_scale(
        calibration_outputs["log_samples"],
        dataset.targets_log_vol[
            split.calibration_indices
        ],
        cfg,
    )
    regime_temperature = (
        calibrate_regime_temperature(
            calibration_outputs["model_logits"],
            split.regime_labels[
                split.calibration_indices
            ],
            cfg,
        )
    )

    print(
        "\nGenerating untouched test forecasts..."
    )
    test_outputs = collect_ensemble_outputs(
        models,
        dataset,
        split,
        split.test_indices,
        cfg,
    )
    predictions = build_prediction_frame(
        test_outputs,
        dataset,
        split,
        split.test_indices,
        interval_scale,
        regime_temperature,
        har_predictions,
        cfg,
    )

    evaluation = evaluate_predictions(
        predictions,
        cfg,
    )

    print("\nPooled and per-ticker point metrics:")
    print(
        evaluation["point_metrics"]
        .round(5)
        .to_string(index=False)
    )

    print("\nProbabilistic test summary:")
    print(
        evaluation["probabilistic_summary"]
        .round(5)
        .to_string(index=False)
    )

    print("\nRaw test-regime metrics:")
    print(
        evaluation["raw_regime_metrics"]
        .round(5)
        .to_string(index=False)
    )

    print("\nPast-only smoothed test-regime metrics:")
    print(
        evaluation["smoothed_regime_metrics"]
        .round(5)
        .to_string(index=False)
    )

    if evaluation["paired_bootstrap"] is not None:
        print(
            "\nPaired date-block bootstrap differences "
            "(negative favours SigFlow):"
        )
        print(
            evaluation["paired_bootstrap"]
            .round(5)
            .to_string(index=False)
        )

    print("\nActual data source:", dataset.data_source)
    if dataset.data_source.startswith("synthetic"):
        print(
            "WARNING: synthetic results validate the pipeline only."
        )

    save_results(
        models,
        histories,
        predictions,
        evaluation,
        dataset,
        split,
        interval_scale,
        regime_temperature,
        cfg,
    )
    plot_results(
        histories,
        predictions,
        evaluation,
        cfg,
    )

    print("\nComplete.")

    return {
        "dataset": dataset,
        "split": split,
        "models": models,
        "histories": histories,
        "predictions": predictions,
        "evaluation": evaluation,
        "interval_scale": interval_scale,
        "regime_temperature": regime_temperature,
    }


RESULTS = main(CFG)